<a href="https://colab.research.google.com/github/sindhubhargavee/Final-Project-1/blob/main/FinalProj_Bank_FD_Pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
from pathlib import Path

In [3]:
# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               VotingClassifier, StackingClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix, roc_curve)


In [4]:
# Advanced boosting
!pip install catboost
!pip install xgboost
!pip install lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.8 MB/s eta 0:00:00


In [5]:
# Imbalanced learning
!pip install imblearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 8.7 MB/s eta 0:00:00


In [6]:
# SHAP
!pip install shap
import shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 498.0/498.0 kB 16.1 MB/s eta 0:00:00


In [7]:
# Output dir
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

In [8]:
print("=" * 70)
print("1. LOADING DATA")
print("=" * 70)

df = pd.read_csv("/content/bank-additional-full.csv", sep=";")
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nTarget distribution:\n{df['y'].value_counts()}")
vc = df['y'].value_counts()
print(f"\nClass imbalance ratio: {vc['no'] / vc['yes']:.2f}:1")


1. LOADING DATA
Dataset shape: (41188, 21)

Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']

Target distribution:
y
no     36548
yes     4640
Name: count, dtype: int64

Class imbalance ratio: 7.88:1


In [9]:
# 2. EXPLORATORY DATA ANALYSIS
print("\n" + "=" * 70)
print("2. EXPLORATORY DATA ANALYSIS")
print("=" * 70)

print("\nData Types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nBasic stats:\n", df.describe())


2. EXPLORATORY DATA ANALYSIS

Data Types:
 age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object

Missing values:
 age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         

In [10]:
# EDA Plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("EDA: Target Distribution & Key Features", fontsize=14, fontweight="bold")

Text(0.5, 0.98, 'EDA: Target Distribution & Key Features')

In [11]:
# Target distribution
df["y"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["#4C72B0", "#DD8452"])
axes[0, 0].set_title("Target Distribution (y)")
axes[0, 0].set_xlabel("Subscribed")
axes[0, 0].tick_params(axis="x", rotation=0)

In [12]:
# Age distribution by target
df.groupby("y")["age"].plot(kind="kde", ax=axes[0, 1])
axes[0, 1].set_title("Age Distribution by Target")
axes[0, 1].legend(["No", "Yes"])

In [13]:
# Job vs subscription rate
job_rate = df.groupby("job")["y"].apply(lambda x: (x == "yes").mean()).sort_values()
job_rate.plot(kind="barh", ax=axes[0, 2], color="#4C72B0")
axes[0, 2].set_title("Subscription Rate by Job")

Text(0.5, 1.0, 'Subscription Rate by Job')

In [14]:
# Education vs subscription rate
edu_rate = df.groupby("education")["y"].apply(lambda x: (x == "yes").mean()).sort_values()
edu_rate.plot(kind="bar", ax=axes[1, 0], color="#DD8452")
axes[1, 0].set_title("Subscription Rate by Education")
axes[1, 0].tick_params(axis="x", rotation=45)

In [15]:
# Duration histogram
axes[1, 1].hist(df["duration"], bins=50, color="#55A868", edgecolor="white")
axes[1, 1].set_title("Call Duration Distribution")
axes[1, 1].set_xlabel("Duration (seconds)")

Text(0.5, 80.7222222222222, 'Duration (seconds)')

In [16]:
# Month subscription rate
month_rate = df.groupby("month")["y"].apply(lambda x: (x == "yes").mean())
month_order = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]
month_rate = month_rate.reindex([m for m in month_order if m in month_rate.index])
month_rate.plot(kind="bar", ax=axes[1, 2], color="#C44E52")
axes[1, 2].set_title("Subscription Rate by Month")
axes[1, 2].tick_params(axis="x", rotation=45)

In [17]:
plt.tight_layout()
plt.savefig(OUT / "eda_overview.png", dpi=150, bbox_inches="tight")
plt.close()
print("EDA plot saved.")

EDA plot saved.


In [18]:
# Correlation heatmap (numeric)
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
fig, ax = plt.subplots(figsize=(12, 8))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax,
            linewidths=0.5, annot_kws={"size": 8})
ax.set_title("Correlation Heatmap (Numeric Features)", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Correlation heatmap saved.")

Correlation heatmap saved.


In [19]:
# 3. PREPROCESSING & FEATURE ENGINEERING
print("\n" + "=" * 70)
print("3. PREPROCESSING & FEATURE ENGINEERING")
print("=" * 70)

data = df.copy()
# --- Feature Engineering ---
# 1. Age groups
data["age_group"] = pd.cut(data["age"], bins=[0, 25, 35, 45, 55, 100],
                            labels=["18-25", "26-35", "36-45", "46-55", "55+"])

# 2. Has previous contact
data["has_prev_contact"] = (data["pdays"] != 999).astype(int)

# 3. Interaction: duration × campaign (engagement intensity)
data["duration_per_campaign"] = data["duration"] / (data["campaign"] + 1)

# 4. Balance proxy from economic indicators
data["economic_stress"] = data["emp.var.rate"] * data["euribor3m"]

# 5. Ordinal encode education
edu_order = ["illiterate", "basic.4y", "basic.6y", "basic.9y",
             "high.school", "professional.course", "university.degree", "unknown"]
data["education_ord"] = data["education"].map(
    {e: i for i, e in enumerate(edu_order)}).fillna(0).astype(int)

print("Feature engineering complete.")
print(f"New features: age_group, has_prev_contact, duration_per_campaign, "
      f"economic_stress, education_ord")


3. PREPROCESSING & FEATURE ENGINEERING
Feature engineering complete.
New features: age_group, has_prev_contact, duration_per_campaign, economic_stress, education_ord


In [20]:
# --- Define feature groups ---
TARGET = "y"
DROP_COLS = ["education"]  # replaced by ordinal version

# Check if columns to drop exist before dropping them
existing_drop_cols = [col for col in DROP_COLS if col in data.columns]
if existing_drop_cols:
    data = data.drop(columns=existing_drop_cols)
else:
    print(f"Columns {DROP_COLS} not found in data, skipping drop operation.")

NUMERIC_FEATURES = [
    "age", "duration", "campaign", "pdays", "previous",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed",
    "duration_per_campaign", "economic_stress", "education_ord", "has_prev_contact"
]

CATEGORICAL_FEATURES = [
    "job", "marital", "default", "housing", "loan",
    "contact", "month", "day_of_week", "poutcome", "age_group"
]

X = data[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = data[TARGET]

print(f"\nFeature matrix shape: {X.shape}")


Feature matrix shape: (41188, 24)


In [21]:
# 4. TRAIN-TEST SPLIT

# Convert target variable 'y' to numerical (0 and 1)
y = y.map({'no': 0, 'yes': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.3f}, Test: {y_test.mean():.3f}")


Train: (32950, 24), Test: (8238, 24)
Train positive rate: 0.113, Test: 0.113


In [22]:
# 5. PREPROCESSING PIPELINE
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES)
])

In [23]:
# 6. BASELINE MODELS
print("\n" + "=" * 70)
print("6. BASELINE MODELS (Stratified 5-Fold CV)")
print("=" * 70)

BASELINE_MODELS = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "KNN":                 KNeighborsClassifier(n_neighbors=7),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Naive Bayes":         GaussianNB(),
    "SVM":                 SVC(probability=True, random_state=42, kernel="linear", C=1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_results = {}
# Prepare data for CV (preprocess once)
X_train_pre = preprocessor.fit_transform(X_train, y_train)
X_test_pre  = preprocessor.transform(X_test)

for name, model in BASELINE_MODELS.items():
    scores = cross_val_score(model, X_train_pre, y_train,
                             cv=cv, scoring="roc_auc", n_jobs=-1)
    baseline_results[name] = {"CV_AUC_mean": scores.mean(), "CV_AUC_std": scores.std()}
    print(f"  {name:25s}  AUC: {scores.mean():.4f} ± {scores.std():.4f}")


6. BASELINE MODELS (Stratified 5-Fold CV)
  Logistic Regression        AUC: 0.9336 ± 0.0041
  Decision Tree              AUC: 0.9239 ± 0.0073
  KNN                        AUC: 0.8859 ± 0.0076
  Random Forest              AUC: 0.9438 ± 0.0044
  Naive Bayes                AUC: 0.8491 ± 0.0045
  SVM                        AUC: 0.9318 ± 0.0050


In [24]:
# 7. ADVANCED MODELS
print("\n" + "=" * 70)
print("7. ADVANCED MODELS (with SMOTE for imbalance)")
print("=" * 70)
# Apply SMOTE on training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_pre, y_train)
print(f"After SMOTE — Train shape: {X_train_sm.shape}, "
      f"Positive rate: {y_train_sm.mean():.3f}")

ADVANCED_MODELS = {
    "XGBoost": XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric="logloss",
        random_state=42, n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=80, max_depth=4, learning_rate=0.1,
        subsample=0.8, random_state=42
    ),
}

advanced_results = {}
trained_models = {}

for name, model in ADVANCED_MODELS.items():
    scores = cross_val_score(model, X_train_sm, y_train_sm,
                             cv=cv, scoring="roc_auc", n_jobs=-1)
    model.fit(X_train_sm, y_train_sm)
    y_pred      = model.predict(X_test_pre)
    y_prob      = model.predict_proba(X_test_pre)[:, 1]
    test_auc    = roc_auc_score(y_test, y_prob)
    test_f1     = f1_score(y_test, y_pred)
    advanced_results[name] = {
        "CV_AUC_mean": scores.mean(), "CV_AUC_std": scores.std(),
        "Test_AUC": test_auc, "Test_F1": test_f1
    }
    trained_models[name] = model
    print(f"  {name:25s}  CV-AUC: {scores.mean():.4f} ± {scores.std():.4f}"
          f"  |  Test-AUC: {test_auc:.4f}  F1: {test_f1:.4f}")



7. ADVANCED MODELS (with SMOTE for imbalance)
After SMOTE — Train shape: (58476, 64), Positive rate: 0.500
  XGBoost                    CV-AUC: 0.9922 ± 0.0004  |  Test-AUC: 0.9522  F1: 0.6754
  LightGBM                   CV-AUC: 0.9924 ± 0.0004  |  Test-AUC: 0.9517  F1: 0.6667
  GradientBoosting           CV-AUC: 0.9898 ± 0.0005  |  Test-AUC: 0.9508  F1: 0.6643


In [25]:
# 8. STACKING ENSEMBLE
print("\n" + "=" * 70)
print("8. STACKING ENSEMBLE")
print("=" * 70)
estimators = [
    ("xgb", XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1,
                           use_label_encoder=False, eval_metric="logloss",
                           random_state=42, n_jobs=-1)),
    ("lgbm", LGBMClassifier(n_estimators=80, max_depth=4, learning_rate=0.1,
                             random_state=42, n_jobs=-1, verbose=-1)),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
]
meta_learner = LogisticRegression(max_iter=500)

stacking = StackingClassifier(estimators=estimators, final_estimator=meta_learner,
                               cv=3, n_jobs=-1, passthrough=False)
stacking.fit(X_train_sm, y_train_sm)

y_stack_pred = stacking.predict(X_test_pre)
y_stack_prob = stacking.predict_proba(X_test_pre)[:, 1]
stack_auc  = roc_auc_score(y_test, y_stack_prob)
stack_f1   = f1_score(y_test, y_stack_pred)
print(f"  Stacking Ensemble  →  Test-AUC: {stack_auc:.4f}  F1: {stack_f1:.4f}")



8. STACKING ENSEMBLE
  Stacking Ensemble  →  Test-AUC: 0.9469  F1: 0.6126


In [26]:
# 9. FULL EVALUATION & LEADERBOARD
print("\n" + "=" * 70)
print("9. FULL EVALUATION — LEADERBOARD")
print("=" * 70)
# Fit baseline models on SMOTE data too and evaluate on test
all_results = []

for name, model in {**BASELINE_MODELS, **ADVANCED_MODELS}.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test_pre)
    y_prob = model.predict_proba(X_test_pre)[:, 1]
    all_results.append({
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall":    recall_score(y_test, y_pred),
        "F1-Score":  f1_score(y_test, y_pred),
        "ROC-AUC":   roc_auc_score(y_test, y_prob),
    })

all_results.append({
    "Model":     "Stacking Ensemble",
    "Accuracy":  accuracy_score(y_test, y_stack_pred),
    "Precision": precision_score(y_test, y_stack_pred, zero_division=0),
    "Recall":    recall_score(y_test, y_stack_pred),
    "F1-Score":  f1_score(y_test, y_stack_pred),
    "ROC-AUC":   roc_auc_score(y_test, y_stack_prob),
})

leaderboard = pd.DataFrame(all_results).sort_values("ROC-AUC", ascending=False)
leaderboard = leaderboard.reset_index(drop=True)
leaderboard.index += 1
print(leaderboard.to_string())
leaderboard.to_csv(OUT / "leaderboard.csv", index=True)



9. FULL EVALUATION — LEADERBOARD
                  Model  Accuracy  Precision    Recall  F1-Score   ROC-AUC
1               XGBoost  0.919034   0.615794  0.747845  0.675426  0.952225
2              LightGBM  0.918063   0.615314  0.727371  0.666667  0.951732
3         Random Forest  0.917820   0.623645  0.682112  0.651570  0.951000
4      GradientBoosting  0.907502   0.561848  0.812500  0.664317  0.950771
5     Stacking Ensemble  0.915878   0.636469  0.590517  0.612633  0.946914
6   Logistic Regression  0.867444   0.455483  0.904095  0.605776  0.943465
7                   SVM  0.850328   0.424914  0.929957  0.583305  0.942394
8         Decision Tree  0.859432   0.438568  0.884698  0.586429  0.931471
9                   KNN  0.843530   0.409251  0.877155  0.558108  0.906509
10          Naive Bayes  0.802986   0.336317  0.769397  0.468043  0.845172


In [27]:
# 10. BEST MODEL — DETAILED EVALUATION
print("\n" + "=" * 70)
print("10. BEST MODEL — DETAILED EVALUATION")
print("=" * 70)

best_name = leaderboard.iloc[0]["Model"]
print(f"\nBest model: {best_name}")

if best_name == "Stacking Ensemble":
    best_model = stacking
    y_best_pred = y_stack_pred
    y_best_prob = y_stack_prob
else:
    best_model = {**BASELINE_MODELS, **ADVANCED_MODELS}[best_name]
    y_best_pred = best_model.predict(X_test_pre)
    y_best_prob = best_model.predict_proba(X_test_pre)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_best_pred, target_names=["No", "Yes"]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_best_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
axes[0].set_title(f"Confusion Matrix — {best_name}", fontsize=11)
axes[0].set_ylabel("Actual")
axes[0].set_xlabel("Predicted")

# ROC Curve — all top models
for _, row in leaderboard.iterrows():
    nm = row["Model"]
    if nm == "Stacking Ensemble":
        prob = y_stack_prob
    else:
        m = {**BASELINE_MODELS, **ADVANCED_MODELS}.get(nm)
        if m is None:
            continue
        prob = m.predict_proba(X_test_pre)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[1].plot(fpr, tpr, label=f"{nm} ({row['ROC-AUC']:.3f})", linewidth=1.5)

axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curves — All Models", fontsize=11)
axes[1].legend(loc="lower right", fontsize=7)

plt.tight_layout()
plt.savefig(OUT / "best_model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("Evaluation plots saved.")



10. BEST MODEL — DETAILED EVALUATION

Best model: XGBoost

Classification Report:
              precision    recall  f1-score   support

          No       0.97      0.94      0.95      7310
         Yes       0.62      0.75      0.68       928

    accuracy                           0.92      8238
   macro avg       0.79      0.84      0.81      8238
weighted avg       0.93      0.92      0.92      8238

Evaluation plots saved.


In [28]:
# 11. FEATURE IMPORTANCE & SHAP
print("\n" + "=" * 70)
print("11. FEATURE IMPORTANCE & SHAP EXPLAINABILITY")
print("=" * 70)

# Use XGBoost for SHAP (most interpretable)
xgb_model = trained_models.get("XGBoost")
if xgb_model is None:
    xgb_model = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1,
                               use_label_encoder=False, eval_metric="logloss",
                               random_state=42, n_jobs=-1)
    xgb_model.fit(X_train_sm, y_train_sm)

# Feature names after preprocessing
ohe_cats = (preprocessor.named_transformers_["cat"]
            .named_steps["onehot"]
            .get_feature_names_out(CATEGORICAL_FEATURES).tolist())
feature_names = NUMERIC_FEATURES + ohe_cats

# Built-in feature importance
importances = pd.Series(xgb_model.feature_importances_, index=feature_names)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20.sort_values().plot(kind="barh", ax=ax, color="#3498db")
ax.set_title("XGBoost — Top 20 Feature Importances", fontsize=12, fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig(OUT / "feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("Feature importance plot saved.")

# SHAP analysis (on a sample for speed)
print("\nComputing SHAP values (sample of 500 test instances)...")
sample_idx = np.random.choice(len(X_test_pre), size=min(200, len(X_test_pre)), replace=False)
X_shap = X_test_pre[sample_idx]

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)

# SHAP Summary plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
                  max_display=20, show=False)
plt.title("SHAP Summary Plot — Feature Impact on Predictions", fontsize=12)
plt.tight_layout()
plt.savefig(OUT / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.close()
print("SHAP summary plot saved.")

# SHAP Bar plot (mean absolute)
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
                  plot_type="bar", max_display=20, show=False)
plt.title("SHAP Feature Importance (Mean |SHAP|)", fontsize=12)
plt.tight_layout()
plt.savefig(OUT / "shap_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("SHAP bar plot saved.")




11. FEATURE IMPORTANCE & SHAP EXPLAINABILITY
Feature importance plot saved.

Computing SHAP values (sample of 500 test instances)...
SHAP summary plot saved.
SHAP bar plot saved.


In [29]:
# 12. SUMMARY
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"\nBest Model  : {best_name}")
print(f"Test ROC-AUC: {leaderboard.iloc[0]['ROC-AUC']:.4f}")
print(f"Test F1     : {leaderboard.iloc[0]['F1-Score']:.4f}")
print(f"Test Recall : {leaderboard.iloc[0]['Recall']:.4f}")
print(f"\nAll outputs saved to: {OUT.resolve()}")
print("\nFiles generated:")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name}")
print("\n✅ Pipeline complete!")



SUMMARY

Best Model  : XGBoost
Test ROC-AUC: 0.9522
Test F1     : 0.6754
Test Recall : 0.7478

All outputs saved to: /content/outputs

Files generated:
  best_model_evaluation.png
  correlation_heatmap.png
  eda_overview.png
  feature_importance.png
  leaderboard.csv
  shap_bar.png
  shap_summary.png

✅ Pipeline complete!
